# ACE trajectories example


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import importlib

# Define project root (run from MHDTurbPy repository root)
root_dir = str(Path.cwd())

sc_pos_path = Path(root_dir).joinpath("functions", "sc_pos")
sys.path.insert(0, str(sc_pos_path))

import interactive_orbits_timeseries_plus3d as orbits
importlib.reload(orbits)


## Build the 3D trajectory figure


In [ ]:
targets = ["ACE", "WIND", "PSP"]

fig_3d = orbits.build_3d_figure(
    targets=targets,
    start="2021-10-01T00:00:00",
    stop="2021-12-10T00:00:00",
    step="6h",
    frame3d="HCI",
    rss_rsun=20,
    skip_small_bodies=True,
)

fig_3d.show()


## Build synchronized longitude/latitude time series


In [ ]:
fig_ts = orbits.build_timeseries_figure(
    targets=targets,
    start="2021-10-01T00:00:00",
    stop="2021-12-10T00:00:00",
    step="6h",
    lon_unwrap_deg=False,
)

fig_ts.show()


## MRE: first-principles stream-separation metric (step by step)

This minimal reproducible example (MRE) intentionally keeps a few assumptions fixed so the method is transparent:

- **Hardcoded/assumed in this notebook run:** `flow_dir_gse=(-1,0,0)`, constant `vsw_kms=400`, `perp_scale=0.35`, `lag_tolerance=0.5`.
- **Why this is limited:** real solar wind has time-varying speed and flow direction, and anisotropy changes with conditions.
- **How to improve with real data:** replace constants by measured `Vsw(t)`, infer local flow direction from plasma moments, and make decorrelation scales time dependent.

The solver minimizes a conservative estimate of the probability that **all** selected spacecraft observe the same stream.


In [ ]:
# Step 1: configure assumptions explicitly (so they are visible and reproducible)
metric_cfg = dict(
    targets=targets,
    start="2021-10-01T00:00:00",
    stop="2021-12-10T00:00:00",
    step="3h",
    window_hours=18,
    top_n=3,
    flow_dir_gse=(-1.0, 0.0, 0.0),   # hardcoded Parker-like anti-sunward direction
    vsw_kms=400.0,                    # hardcoded constant speed [km/s]
    perp_scale=0.35,                  # hardcoded decorrelation scale fraction
    lag_tolerance=0.5,                # hardcoded lag tolerance fraction
    along_weight=0.25,
    min_coverage=0.8,
    verbose=True,
)

# Step 2: run first-principles window ranking
scores_out = orbits.find_best_stream_aligned_intervals(**metric_cfg)
best_windows = scores_out['best']
all_scores = scores_out['scores']

# Step 3: inspect numerical outputs with uncertainty bars (16th-84th percentile)
cols = [
    'window_start', 'window_end', 'alignment_metric', 'alignment_metric_p16', 'alignment_metric_p84',
    'same_stream_prob', 'same_stream_prob_p16', 'same_stream_prob_p84',
    'same_flow_score', 'same_flow_score_p16', 'same_flow_score_p84',
]
display(best_windows[cols])

# Step 4: build intuitive visualization for each selected interval
alignment_figs = []
for _, row in best_windows.iterrows():
    alignment_figs.append(
        orbits.plot_stream_alignment_interval(
            tracks=scores_out['tracks'],
            targets=targets,
            window_start=row['window_start'],
            window_end=row['window_end'],
            flow_hat=scores_out['flow_hat'],
            summary_row=row,
        )
    )

for fig in alignment_figs:
    fig.show()
